# 2. Data Collection - Web Scraping
Scrape the Falcon 9 launch tables from Wikipedia (fixed page revision) with BeautifulSoup.

In [ ]:
import requests
import pandas as pd
from io import StringIO
from bs4 import BeautifulSoup

url = 'https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922'
headers = {'User-Agent': 'Mozilla/5.0 (capstone project)'}
response = requests.get(url, headers=headers)
print('status:', response.status_code)
soup = BeautifulSoup(response.text, 'html.parser')
print(soup.title.string)

In [ ]:
# Launch tables have this CSS class
tables = soup.find_all('table', 'wikitable plainrowheaders collapsible')
print('launch tables found:', len(tables))
frames = []
for t in tables:
    try:
        frames.append(pd.read_html(StringIO(str(t)))[0])
    except ValueError:
        pass
print('tables parsed:', len(frames))
frames[0].head()

In [ ]:
# Keep tables that share the standard 'Flight No.' layout and stack them
cleaned = []
for f in frames:
    f = f.copy()
    f.columns = [str(c).split('[')[0].strip() for c in f.columns]
    if 'Flight No.' in f.columns:
        cleaned.append(f)
launches = pd.concat(cleaned, ignore_index=True)
launches = launches.dropna(how='all')
print(launches.shape)
launches.head()

In [ ]:
launches.to_csv('../data/spacex_web_scraped.csv', index=False)
print('saved')